# 二维卷积（Conv2D）

二维卷积在输入特征图上滑动卷积核，聚合局部空间和通道信息，并生成新的特征图。深度学习框架中的 Conv2D 通常实现的是互相关（不翻转卷积核）。

设输入为 $X\in\mathbb{R}^{N\times C_{in}\times H\times W}$，卷积核为 $W\in\mathbb{R}^{C_{out}\times C_{in}\times K_h\times K_w}$，则：

$$Y_{n,c_o,h,w}=b_{c_o}+\sum_{c_i=1}^{C_{in}}\sum_{u=0}^{K_h-1}\sum_{v=0}^{K_w-1}W_{c_o,c_i,u,v}X_{n,c_i,h\cdot s+u-p,w\cdot s+v-p}.$$

其中 $s$ 为步幅、$p$ 为填充大小。无膨胀时输出空间尺寸为：

$$H_{out}=\left\lfloor\frac{H+2p-K_h}{s}\right\rfloor+1,\qquad W_{out}=\left\lfloor\frac{W+2p-K_w}{s}\right\rfloor+1.$$

下方代码使用 `unfold`（im2col）将局部窗口展开，再通过矩阵乘法完成卷积计算。

In [ ]:
import torch
import torch.nn.functional as F

def conv2d_manual(x, weight, bias=None, stride=1, padding=0):
    """
    x: [N, C_in, H, W]，分别是批大小、输入通道、高、宽。
    weight: [C_out, C_in, K, K]；每个输出通道对应一个跨所有输入通道的卷积核。
    bias: 可选 [C_out]。返回 [N, C_out, H_out, W_out]。
    """
    N, C_in, H, W = x.shape
    C_out, _, K, _ = weight.shape
    
    # 1. 计算滑窗位置数，即输出特征图的空间形状 [H_out, W_out]。
    H_out = (H + 2 * padding - K) // stride + 1
    W_out = (W + 2 * padding - K) // stride + 1
    
    # 2. F.pad 只扩展最后两个空间维；填充后 x 为 [N, C_in, H+2p, W+2p]。
    if padding > 0:
        x = F.pad(x, (padding, padding, padding, padding))
        
    # 3. 使用 im2col (unfold) 将输入转换为矩阵
    # unfold 将滑动窗口展开为 [N, C_in * K * K, L] 其中 L = H_out * W_out
    x_unfolded = F.unfold(x, kernel_size=K, stride=stride)  # [N, C_in*K*K, L]
    
    # 4. 每个卷积核展平为长度 C_in*K*K 的向量。
    weight_flat = weight.view(C_out, -1)  # [C_out, C_in*K*K]
    
    # 5. 矩阵乘法 (卷积转化为矩阵乘)
    # 先转置窗口维： [N, C_in*K*K, L] -> [N, L, C_in*K*K]。
    # 再与 [C_in*K*K, C_out] 相乘，得到每个空间位置的所有输出通道：[N, L, C_out]。
    out = torch.matmul(x_unfolded.transpose(1, 2), weight_flat.t())  # [N, L, C_out]
    
    # 6. 将偏置重塑为 [1, 1, C_out]，自动广播至 N 个样本和 L 个空间位置。
    if bias is not None:
        out += bias.view(1, 1, -1)
        
    # [N, L, C_out] -> [N, C_out, L]；L=H_out*W_out，再恢复二维空间布局。
    out = out.transpose(1, 2).view(N, C_out, H_out, W_out)
    return out